# PyTorch 第九章：自然语言处理项目案例

> 对应《PyTorch 实用教程（第二版）》第九章  
> 目标：用**最小可运行实验**理解 NLP 的文本数值化、RNN/LSTM、Seq2Seq、Transformer、BERT 与 GPT，并建立通往 LLM 的核心知识链。

## 本章覆盖顺序

1. 9.1 自然语言处理简介
2. 9.2 文本分类——RNN / LSTM
3. 9.3 机器翻译——Seq2Seq
4. 9.4 机器翻译——Transformer
5. 9.5 命名实体识别——BERT
6. 9.6 文章续写 / 问答对话——GPT

教程章节入口：  
https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-9/

### 本 Notebook 的取舍

- 不下载 IMDB、Tatoeba、CLUENER、中文 GPT-2 等大型数据或权重。
- 使用少量内置文本 / 随机张量构造等价实验。
- 不复制大段工程源码，只保留关键数据流、shape、loss、mask 与推理逻辑。
- 重点加强与后续 Transformer / LLM 直接相关的内容。

## 0. 环境导入

In [ ]:
import math
import random
import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("device:", device)

## 当前接口与教程差异

本章教程部分代码来自较早版本生态，因此需要注意：

- PyTorch 的 `nn.RNN`、`nn.LSTM`、`nn.Transformer*` 仍可直接使用；本 Notebook 统一使用 `batch_first=True`，shape 更直观。
- Hugging Face 当前更推荐 `AutoTokenizer`、`AutoModelForTokenClassification`、`AutoModelForCausalLM` 等统一接口。
- 当前 Hugging Face 因果语言模型通常可直接传 `labels=input_ids`，模型内部完成右移；忽略标签使用 `-100`，不是旧代码里的 `-1`。
- GPT 属于 **Causal Language Modeling（因果语言建模）**，不是 BERT 式 Masked Language Modeling。

# 9.1 自然语言处理简介

原教程顺序：

1. NLP 的主要难点
2. NLP 任务处理流程
3. 常见 NLP 任务
4. NLP 基础概念

NLP 的核心难点包括：

- 文本本身不是数值，需要 tokenization + numericalization；
- 语义依赖上下文；
- 存在长距离依赖；
- 同一个 token 在不同上下文中可能含义不同。

## 文本进入神经网络前发生了什么？

最小流程：

`raw text → tokenize → vocabulary/tokenizer → token ids → embedding → model`

现代 LLM 的 tokenizer 往往使用 subword / byte-level 方法，而不是简单按空格切词；但“token → id → embedding”这个主流程没有变化。

In [ ]:
texts = [
    "this movie is good",
    "this movie is bad",
    "good acting and good music",
]

def simple_tokenize(text):
    return re.findall(r"[a-z]+|[.!?]", text.lower())

tokenized = [simple_tokenize(t) for t in texts]
print(tokenized)

## 词表 Vocabulary：token 与整数 id 的映射

保留特殊 token：

- `<pad>`：补齐 batch
- `<unk>`：未知 token
- `<bos>`：生成开始
- `<eos>`：生成结束

In [ ]:
special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]

counter = Counter(tok for sent in tokenized for tok in sent)
vocab = {tok: i for i, tok in enumerate(special_tokens)}

for tok, _ in counter.most_common():
    if tok not in vocab:
        vocab[tok] = len(vocab)

print(vocab)

In [ ]:
PAD_ID = vocab["<pad>"]
UNK_ID = vocab["<unk>"]

def encode(tokens, max_len):
    ids = [vocab.get(tok, UNK_ID) for tok in tokens][:max_len]
    ids += [PAD_ID] * (max_len - len(ids))
    return ids

batch_ids = torch.tensor([encode(x, 6) for x in tokenized])
print(batch_ids)
print("shape:", batch_ids.shape)

## Embedding：不是 one-hot，而是“查表”

`nn.Embedding(V, D)` 可以看成一个 `[V, D]` 的可学习矩阵。

输入 token id 后，PyTorch 直接取对应行作为 token vector。

In [ ]:
embedding = nn.Embedding(
    num_embeddings=len(vocab),
    embedding_dim=8,
    padding_idx=PAD_ID,
)

x_embed = embedding(batch_ids)

print("token ids:", batch_ids.shape)
print("embedding:", x_embed.shape)
print("PAD embedding:", embedding.weight[PAD_ID])

## NLP 任务的两个大类

为了学习模型结构，可以先粗略分为：

- **sequence → class**：情感分类、文本分类
- **sequence → sequence**：翻译、摘要、生成

NER 更准确地说是 **token classification**：输入一个序列，每个 token 输出一个类别。

# 9.2 文本分类——RNN / LSTM

原教程核心流程：

1. 文本清洗与分词
2. 统计词频、构建词表
3. Dataset 中完成 token → id 与 padding / truncation
4. `nn.Embedding`
5. `nn.RNN`
6. `nn.LSTM` / BiLSTM
7. 加载 GloVe 预训练 embedding

## RNN：序列状态递推

最基本的 RNN：

$$
h_t=\tanh(W_{xh}x_t+W_{hh}h_{t-1}+b)
$$

它按时间步不断更新 hidden state，因此天然适合序列。

但长序列上容易出现梯度消失 / 爆炸与长期依赖问题。

In [ ]:
B, T, D = 3, 5, 8
H = 12

x = torch.randn(B, T, D)

rnn = nn.RNN(
    input_size=D,
    hidden_size=H,
    num_layers=1,
    batch_first=True,
)

outputs, h_n = rnn(x)

print("outputs:", outputs.shape)  # [B, T, H]
print("h_n:", h_n.shape)          # [layers, B, H]
print("last output == h_n[-1]:", torch.allclose(outputs[:, -1], h_n[-1]))

## 最小 RNN 文本分类器

数据流：

`token ids → Embedding → RNN → final hidden → Linear → class logits`

注意：分类层输出的是 **logits**，不要在模型中手动 `softmax` 后再送给 `CrossEntropyLoss`。

In [ ]:
class RNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.rnn = nn.RNN(
            embed_dim, hidden_dim,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        _, h_n = self.rnn(x)
        return self.fc(h_n[-1])

model_rnn = RNNTextClassifier(
    vocab_size=len(vocab),
    embed_dim=8,
    hidden_dim=12,
    num_classes=2,
    pad_id=PAD_ID,
)

labels = torch.tensor([1, 0, 1])
logits = model_rnn(batch_ids)
loss = F.cross_entropy(logits, labels)

print("logits:", logits.shape)
print("loss:", round(float(loss), 4))

### 一个隐藏问题：直接取最后时间步可能把 `<pad>` 当成真实输入

真实任务中应考虑有效长度，例如：

- `pack_padded_sequence`
- 根据 attention mask / lengths 取最后有效 token
- 对序列表示做 masked pooling

教程为了主线清晰没有展开，这里需要明确知道这个问题存在。

## LSTM：增加 cell state 与门控

LSTM 通过：

- forget gate
- input gate
- output gate

控制长期信息流。

PyTorch 输出：

- `output`: 所有时间步 hidden
- `h_n`: 最终 hidden state
- `c_n`: 最终 cell state

In [ ]:
lstm = nn.LSTM(
    input_size=D,
    hidden_size=H,
    num_layers=1,
    batch_first=True,
)

outputs, (h_n, c_n) = lstm(x)

print("outputs:", outputs.shape)
print("h_n:", h_n.shape)
print("c_n:", c_n.shape)

## BiLSTM：同时读取左右上下文

设置 `bidirectional=True` 后，每层都有前向和后向两个方向。

因此输出最后一维会从 `H` 变为 `2H`。

In [ ]:
bilstm = nn.LSTM(
    input_size=D,
    hidden_size=H,
    batch_first=True,
    bidirectional=True,
)

bi_out, (bi_h, bi_c) = bilstm(x)

print("output:", bi_out.shape)  # [B, T, 2H]
print("h_n:", bi_h.shape)       # [2, B, H]

sentence_repr = torch.cat([bi_h[-2], bi_h[-1]], dim=-1)
print("sentence representation:", sentence_repr.shape)

## 预训练 Embedding：理解机制即可

教程使用 GloVe。核心操作只是：

1. 构造 `[vocab_size, embedding_dim]` 矩阵；
2. 让每一行与自己的 token id 对齐；
3. 拷贝进 `nn.Embedding.weight`。

现代 NLP 中，大多数任务会直接使用 BERT / GPT 一类上下文化预训练模型；静态 GloVe 已不再是主流起点，但这个权重加载机制仍值得理解。

In [ ]:
pretrained_matrix = torch.randn(len(vocab), 8)
pretrained_matrix[PAD_ID].zero_()

emb = nn.Embedding(len(vocab), 8, padding_idx=PAD_ID)
with torch.no_grad():
    emb.weight.copy_(pretrained_matrix)

print("loaded:", torch.allclose(emb.weight, pretrained_matrix))

# 9.3 机器翻译——Seq2Seq

原教程强调四个关键点：

1. `<bos> / <eos> / <unk> / <pad>`
2. 最大序列长度
3. Teacher Forcing
4. 训练与推理逻辑不同

经典 Seq2Seq：

`source → Encoder LSTM → hidden/cell → Decoder LSTM → target tokens`

## 构造一个极小翻译 batch

训练 decoder 时，目标序列通常错开一位：

- decoder input：`<bos> 我 爱 你`
- label：`我 爱 你 <eos>`

In [ ]:
PAD, BOS, EOS = 0, 1, 2
SRC_V = 20
TGT_V = 30

src = torch.tensor([
    [4, 5, 6, EOS, PAD],
    [7, 8, EOS, PAD, PAD],
])

target = torch.tensor([
    [BOS, 10, 11, 12, EOS],
    [BOS, 13, 14, EOS, PAD],
])

decoder_input = target[:, :-1]
decoder_label = target[:, 1:]

print("src:", src.shape)
print("decoder input:", decoder_input)
print("label:", decoder_label)

## Encoder：只需要传递最终状态给 Decoder

In [ ]:
class EncoderLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        outputs, (h, c) = self.lstm(x)
        return outputs, h, c

encoder = EncoderLSTM(SRC_V)
enc_out, h, c = encoder(src)

print("encoder outputs:", enc_out.shape)
print("h:", h.shape)
print("c:", c.shape)

## Decoder：一次生成一个 token

每一步输入：

- 当前 token
- 上一步 hidden
- 上一步 cell

输出：

- 当前词表 logits
- 新 hidden / cell

In [ ]:
class DecoderLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids, h, c):
        x = self.embedding(token_ids).unsqueeze(1)   # [B] -> [B,1,D]
        out, (h, c) = self.lstm(x, (h, c))
        logits = self.fc(out[:, 0])
        return logits, h, c

decoder = DecoderLSTM(TGT_V)

first_token = target[:, 0]  # <bos>
logits, h2, c2 = decoder(first_token, h, c)

print("logits:", logits.shape)

## Teacher Forcing

训练时有真实标签，因此下一步 decoder 输入可以选择：

- ground-truth token
- 上一步模型预测 token

Teacher forcing ratio = 1：总使用真实标签。  
推理时没有真实标签，只能自回归。

In [ ]:
def seq2seq_decode_train(encoder, decoder, src, target, teacher_forcing_ratio=0.5):
    _, h, c = encoder(src)
    B, T = target.shape
    outputs = []
    token = target[:, 0]  # BOS

    for t in range(1, T):
        logits, h, c = decoder(token, h, c)
        outputs.append(logits)
        pred = logits.argmax(dim=-1)

        use_teacher = random.random() < teacher_forcing_ratio
        token = target[:, t] if use_teacher else pred

    return torch.stack(outputs, dim=1)

seq_logits = seq2seq_decode_train(
    encoder, decoder, src, target,
    teacher_forcing_ratio=1.0
)

print("seq logits:", seq_logits.shape)

## Padding 不应参与 loss

`CrossEntropyLoss(ignore_index=PAD)` 是生成任务中的高频写法。

In [ ]:
loss_seq2seq = F.cross_entropy(
    seq_logits.reshape(-1, TGT_V),
    decoder_label.reshape(-1),
    ignore_index=PAD,
)

print("loss:", round(float(loss_seq2seq), 4))

## 推理：真正的自回归生成

从 `<bos>` 开始，每次把自己的预测作为下一步输入，直到：

- 生成 `<eos>`，或
- 达到 `max_new_tokens`

In [ ]:
@torch.inference_mode()
def greedy_translate(encoder, decoder, src, max_new_tokens=6):
    _, h, c = encoder(src)
    token = torch.full((src.size(0),), BOS, dtype=torch.long)
    generated = []

    for _ in range(max_new_tokens):
        logits, h, c = decoder(token, h, c)
        token = logits.argmax(dim=-1)
        generated.append(token)
        if torch.all(token == EOS):
            break

    return torch.stack(generated, dim=1)

generated = greedy_translate(encoder, decoder, src)
print("generated ids:", generated)

## BLEU：教程中的 n-gram 命中率是直观解释，不是完整 BLEU

标准 BLEU 还包括：

- clipped n-gram precision
- 多阶 n-gram 几何平均
- brevity penalty

所以不要把单纯的“n-gram 命中次数 / 总次数”当成完整 BLEU。

# 9.4 机器翻译——Transformer

这是本章最重要的小节之一。

原教程模型结构层级：

1. Transformer = Encoder + Decoder + masks
2. Encoder / Decoder = position encoding + repeated blocks
3. Block = Multi-Head Attention + FFN + residual + LayerNorm
4. Attention 内部 = Scaled Dot-Product Attention

核心公式：

$$
\mathrm{Attention}(Q,K,V)
=
\mathrm{softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}
\right)V
$$

## Q / K / V 到底是什么？

对 self-attention：

- Q、K、V 都来自同一个输入序列，只是经过不同线性投影。
- `QK^T` 得到 token 与 token 之间的相关性。
- softmax 得到权重。
- 权重对 V 做加权平均。

多头注意力只是把表示拆到多个子空间独立计算，再拼接。

In [ ]:
B, T, D = 2, 4, 8
H = 2
HEAD_D = D // H

x = torch.randn(B, T, D)

Wq = nn.Linear(D, D, bias=False)
Wk = nn.Linear(D, D, bias=False)
Wv = nn.Linear(D, D, bias=False)

q = Wq(x).view(B, T, H, HEAD_D).transpose(1, 2)
k = Wk(x).view(B, T, H, HEAD_D).transpose(1, 2)
v = Wv(x).view(B, T, H, HEAD_D).transpose(1, 2)

scores = q @ k.transpose(-2, -1) / math.sqrt(HEAD_D)
attn = scores.softmax(dim=-1)
head_out = attn @ v

print("q:", q.shape)
print("attention matrix:", attn.shape)
print("head output:", head_out.shape)
print("one row sums to:", attn[0, 0, 0].sum().item())

## 为什么除以 $\sqrt{d_k}$？

如果维度增大，点积绝对值通常变大，softmax 会更容易饱和，使梯度变小。

缩放后数值范围更稳定。

## Positional Encoding

Transformer 本身没有递归结构，若完全不加入位置信息，它不知道：

`我 爱 你`

和

`你 爱 我`

在 token 顺序上有什么不同。

原始 Transformer 使用固定 sinusoidal positional encoding。

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # 固定位置编码不是参数，因此注册成 buffer
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

pe = SinusoidalPositionalEncoding(8)
x0 = torch.zeros(2, 5, 8)
x_pos = pe(x0)

print(x_pos.shape)
print("position 0:", x_pos[0, 0])
print("position 1:", x_pos[0, 1])

## 两类 mask 必须分清

### 1. Padding mask

告诉 attention：这些 `<pad>` 不是正文。

### 2. Causal mask

Decoder / GPT 使用。位置 `t` 只能看到 `≤ t` 的 token，不能偷看未来。

训练时虽然整个 target 一次送进 GPU，但 causal mask 保证模型不会“未卜先知”。

In [ ]:
def causal_mask(seq_len):
    # True 表示禁止 attention
    return torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool),
        diagonal=1
    )

mask = causal_mask(5)
print(mask.int())

## PyTorch 官方 Transformer：先看 shape，再看源码

`nn.Transformer` 已封装 Encoder-Decoder 主结构。

真实语言任务还需要：

- token embedding
- positional encoding
- output projection to vocabulary
- padding mask
- causal mask

In [ ]:
d_model = 16

transformer = nn.Transformer(
    d_model=d_model,
    nhead=4,
    num_encoder_layers=1,
    num_decoder_layers=1,
    dim_feedforward=32,
    dropout=0.0,
    batch_first=True,
)

src_embed = torch.randn(2, 5, d_model)
tgt_embed = torch.randn(2, 4, d_model)
tgt_mask = nn.Transformer.generate_square_subsequent_mask(4)

out = transformer(
    src_embed,
    tgt_embed,
    tgt_mask=tgt_mask,
)

print("output:", out.shape)

## Transformer 训练与推理的关键差异

### 训练

target 全序列并行输入，但使用 causal mask；因此所有时间步可并行计算。

### 推理

必须自回归：

`<bos> → token1 → token2 → ... → <eos>`

因此 decoder-only LLM 推理仍然是逐 token 的。

这也是 KV Cache 极其重要的原因：过去 token 的 K/V 不应每一步重复计算。

## 三种注意力位置

经典 Encoder-Decoder Transformer 中：

1. **Encoder self-attention**：Q/K/V 都来自 encoder 当前序列。
2. **Decoder masked self-attention**：Q/K/V 都来自 decoder，但加 causal mask。
3. **Cross-attention**：Q 来自 decoder，K/V 来自 encoder 输出。

GPT 这类 decoder-only 模型没有 encoder 和 cross-attention 主干；BERT 则主要使用 encoder self-attention。

# 9.5 命名实体识别——BERT

原教程顺序：

1. BERT 预训练思想
2. BERT 下游微调
3. NER 与 BIO 标注
4. CLUENER 数据处理
5. tokenizer / input ids / mask / labels
6. BERT token classification
7. 训练与推理

## BERT = 双向 Transformer Encoder

经典 BERT 预训练包含：

- MLM（Masked Language Modeling）
- NSP（Next Sentence Prediction）

MLM 中，选中约 15% token 作为预测目标，再按经典策略做 80% `[MASK]`、10% 随机 token、10% 保留原 token。

但要注意：后续研究表明 NSP 并非所有 BERT 类模型都必须保留，例如 RoBERTa 就移除了 NSP。

## BERT 输入的三类 embedding

经典 BERT 输入表示是三者相加：

1. token embedding
2. position embedding
3. segment / token type embedding

这使同一个 Transformer Encoder 能处理单句和句对任务。

In [ ]:
B, T, D = 2, 6, 12
V = 100

token_ids = torch.randint(0, V, (B, T))
position_ids = torch.arange(T).unsqueeze(0).expand(B, -1)
segment_ids = torch.tensor([
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 0, 0, 0],
])

tok_emb = nn.Embedding(V, D)
pos_emb = nn.Embedding(32, D)
seg_emb = nn.Embedding(2, D)

bert_input = (
    tok_emb(token_ids)
    + pos_emb(position_ids)
    + seg_emb(segment_ids)
)

print("BERT input:", bert_input.shape)

## NER = token classification

例如：

`张 三 在 北 京`

可以标成：

`B-PER I-PER O B-LOC I-LOC`

每个 token 都有一个类别，因此模型输出：

`[batch, seq_len, num_labels]`

In [ ]:
label2id = {
    "O": 0,
    "B-PER": 1, "I-PER": 2,
    "B-LOC": 3, "I-LOC": 4,
}
num_labels = len(label2id)

tokens = ["张", "三", "在", "北", "京"]
labels_ner = ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"]

label_ids = torch.tensor([[label2id[x] for x in labels_ner]])
print(label_ids)

## 用小型 Transformer Encoder 模拟 BERT-NER 数据流

不下载预训练权重，只演示核心结构：

`input ids → embeddings → bidirectional Transformer Encoder → Linear → token logits`

In [ ]:
class TinyBertForTokenClassification(nn.Module):
    def __init__(self, vocab_size, num_labels, d_model=32, max_len=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=64,
            dropout=0.0,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.classifier = nn.Linear(d_model, num_labels)

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)

        # PyTorch key padding mask: True = ignore
        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = ~attention_mask.bool()

        h = self.encoder(
            x,
            src_key_padding_mask=key_padding_mask,
        )
        return self.classifier(h)

ner_model = TinyBertForTokenClassification(
    vocab_size=100,
    num_labels=num_labels,
)

ner_input = torch.tensor([[11, 12, 13, 14, 15]])
ner_mask = torch.ones_like(ner_input)

ner_logits = ner_model(ner_input, ner_mask)
print("NER logits:", ner_logits.shape)

In [ ]:
loss_ner = F.cross_entropy(
    ner_logits.reshape(-1, num_labels),
    label_ids.reshape(-1),
)

loss_ner.backward()

print("NER loss:", round(float(loss_ner), 4))
print("classifier grad:", round(ner_model.classifier.weight.grad.norm().item(), 4))

## 现代 Hugging Face 写法

真实项目不需要自己重写 BERT：

```python
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-chinese")
model = AutoModelForTokenClassification.from_pretrained(
    "google-bert/bert-base-chinese",
    num_labels=num_labels,
)
```

真正容易出错的是 **subword token 与原始标签对齐**：

- 一个词可能被拆成多个 subtoken；
- `[CLS]` / `[SEP]` / padding 不是原始实体 token；
- 不参与监督的位置通常设成 `-100`，让 `CrossEntropyLoss` 忽略。

In [ ]:
# 演示 ignore_index=-100：特殊 token / padding 不计入 token classification loss
logits_demo = torch.randn(1, 5, 3)
labels_demo = torch.tensor([[-100, 1, 2, 0, -100]])

loss_demo = F.cross_entropy(
    logits_demo.view(-1, 3),
    labels_demo.view(-1),
    ignore_index=-100,
)

print("loss:", round(float(loss_demo), 4))

# 9.6 文章续写 / 问答对话——GPT

这是连接本章与后续 LLM 的核心小节。

教程回顾了：

- GPT-1：预训练 + 下游微调
- GPT-2：更大规模预训练与 zero-shot
- GPT-3：规模扩大与 in-context learning
- InstructGPT：SFT + Reward Model + PPO / RLHF

随后用中文 GPT-2 展示：

- 预训练数据拼接
- causal LM loss
- 文章续写
- temperature / top-k / top-p sampling

## 先纠正一个术语

GPT 不是 Masked Language Model。

GPT 的训练目标是：

> 给定左边已有 token，预测下一个 token。

即 **Causal Language Modeling / Next-token Prediction**。

如果 token 序列为：

$$
[x_0,x_1,x_2,x_3]
$$

训练关系是：

- 输入 $x_0$ → 预测 $x_1$
- 输入 $x_0,x_1$ → 预测 $x_2$
- 输入 $x_0,x_1,x_2$ → 预测 $x_3$

统一写成：

$$
\mathcal L
=
-\sum_t
\log p(x_t\mid x_{<t})
$$

In [ ]:
input_ids = torch.tensor([
    [5, 8, 4, 9, 2],
    [7, 3, 6, 1, 2],
])

lm_input = input_ids[:, :-1]
lm_labels = input_ids[:, 1:]

print("input:", lm_input)
print("label:", lm_labels)

## Causal mask：GPT 不能看未来

每个 token 只能 attend 到自己和左边 token。

In [ ]:
T = 6
gpt_mask = torch.triu(
    torch.ones(T, T, dtype=torch.bool),
    diagonal=1
)
print(gpt_mask.int())

## 极简 Decoder-only Transformer LM

为了只依赖 PyTorch，这里用 `TransformerEncoder` + causal mask 实现 GPT 风格 decoder-only 主干。

名称虽然是 `TransformerEncoder`，但只要加 causal mask，它就可以演示 GPT 的 masked self-attention 数据流。

In [ ]:
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, num_layers=2, max_len=64):
        super().__init__()
        self.vocab_size = vocab_size
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=64,
            dropout=0.0,
            batch_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)

        x = self.token_emb(input_ids) + self.pos_emb(pos)

        causal = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=input_ids.device),
            diagonal=1,
        )

        h = self.blocks(x, mask=causal)
        return self.lm_head(h)

VOCAB_SIZE = 50
gpt = TinyCausalLM(VOCAB_SIZE)

tokens = torch.randint(0, VOCAB_SIZE, (4, 8))
logits = gpt(tokens)

print("tokens:", tokens.shape)
print("logits:", logits.shape)

## Next-token loss：logits 与 labels 错开一位

In [ ]:
shift_logits = logits[:, :-1].contiguous()
shift_labels = tokens[:, 1:].contiguous()

loss_gpt = F.cross_entropy(
    shift_logits.view(-1, VOCAB_SIZE),
    shift_labels.view(-1),
)

loss_gpt.backward()

print("shift logits:", shift_logits.shape)
print("shift labels:", shift_labels.shape)
print("loss:", round(float(loss_gpt), 4))

当前 Hugging Face 的 `AutoModelForCausalLM` / `GPT2LMHeadModel` 通常可以直接：

```python
outputs = model(input_ids=input_ids, labels=input_ids)
loss = outputs.loss
```

模型内部完成 shift。需要忽略的 label 使用 `-100`。

## Temperature

对 logits 做：

`logits / temperature`

- `T < 1`：分布更尖锐，输出更确定
- `T > 1`：分布更平坦，随机性更强
- `T = 1`：保持原分布

In [ ]:
logits_sample = torch.tensor([2.0, 1.0, 0.0])

for temp in [0.5, 1.0, 2.0]:
    probs = F.softmax(logits_sample / temp, dim=-1)
    print(f"T={temp}:", [round(float(x), 3) for x in probs])

## Top-k sampling

只保留概率最高的 k 个 token，其余设为负无穷。

In [ ]:
def top_k_filter(logits, k):
    if k <= 0 or k >= logits.numel():
        return logits
    threshold = torch.topk(logits, k).values[-1]
    return logits.masked_fill(logits < threshold, float("-inf"))

demo = torch.tensor([3.0, 2.0, 1.0, 0.0])
print(top_k_filter(demo, k=2))

## Top-p / nucleus sampling

不是固定保留 k 个，而是从高到低累计概率，保留累计概率达到 p 的最小 token 集合。

它会根据当前分布自动改变候选集合大小。

In [ ]:
def top_p_filter(logits, p=0.9):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = F.softmax(sorted_logits, dim=-1)
    cumulative = probs.cumsum(dim=-1)

    remove = cumulative > p
    # 至少保留概率最大的 token
    remove[1:] = remove[:-1].clone()
    remove[0] = False

    filtered = logits.clone()
    filtered[sorted_idx[remove]] = float("-inf")
    return filtered

demo = torch.tensor([4.0, 2.0, 1.0, 0.0])
print("raw probs:", F.softmax(demo, dim=-1))
print("top-p logits:", top_p_filter(demo, p=0.9))

## 最小自回归生成

每一步只使用最后一个位置的 logits 来决定下一个 token。

In [ ]:
@torch.inference_mode()
def generate(model, input_ids, max_new_tokens=5, temperature=1.0, top_k=5):
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        logits = model(generated)
        next_logits = logits[:, -1, :] / temperature

        filtered_rows = []
        for row in next_logits:
            filtered_rows.append(top_k_filter(row, top_k))
        next_logits = torch.stack(filtered_rows)

        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token], dim=1)

    return generated

prompt = torch.tensor([[1, 5, 9]])
generated = generate(
    gpt,
    prompt,
    max_new_tokens=5,
    temperature=0.8,
    top_k=5,
)

print("generated ids:", generated)

## GPT 到现代 LLM：真正应该保留的主线

现代 LLM 结构细节已经远超早期 GPT-2，但主干仍可追溯到：

`tokenizer → token embedding → positional information → repeated causal Transformer blocks → LM head → next-token prediction`

后续模型主要在这些方向改进：

- RoPE / ALiBi 等位置编码
- RMSNorm
- SwiGLU / gated FFN
- FlashAttention
- KV Cache
- grouped-query / multi-query attention
- instruction tuning
- preference optimization / RLHF
- tool use / agent

# 章末知识结构总结

## 1. 文本数值化

`text → token → id → embedding`

这是所有 NLP / LLM 的入口。

## 2. RNN / LSTM

顺序计算，用 hidden state 携带历史信息；LSTM 用门控缓解长期依赖问题。

## 3. Seq2Seq

Encoder 压缩输入语义，Decoder 自回归生成；训练常使用 teacher forcing。

## 4. Transformer

用 attention 建模 token 间关系，并通过位置编码补充顺序信息。训练可并行，生成推理仍自回归。

## 5. BERT

双向 Transformer Encoder + MLM 预训练；非常适合分类、NER 等理解型任务。

## 6. GPT

Causal Transformer + next-token prediction；其生成机制是今天大语言模型的直接基础。

# 学完必须会回答的 12 个问题

1. token、token id、embedding 三者分别是什么？
2. `<pad>`、`<unk>`、`<bos>`、`<eos>` 分别解决什么问题？
3. `nn.RNN` 返回的 `output` 和 `h_n` 有什么区别？
4. LSTM 为什么比普通 RNN 更适合长期依赖？`h` 与 `c` 各是什么？
5. 双向 LSTM 为什么输出维度通常是 `2 × hidden_size`？
6. Seq2Seq 中 teacher forcing 是什么？训练和推理为何不能完全相同？
7. 为什么生成任务计算 loss 时常需要 `ignore_index=PAD_ID`？
8. Self-Attention 中 Q、K、V 分别起什么作用？为什么要除以 `sqrt(d_k)`？
9. padding mask 与 causal mask 有什么区别？
10. BERT 为什么是双向模型，而 GPT 必须使用 causal mask？
11. NER 为什么是 token classification？subword tokenizer 会带来什么标签对齐问题？
12. temperature、top-k、top-p 分别怎样改变 GPT 的生成分布？

# 面向后续 LLM 学习的复习优先级

如果目标是继续学习 Transformer / LLM，建议按以下顺序复习：

1. **9.4 Transformer：Q/K/V、Multi-Head Attention、mask**
2. **9.6 GPT：causal LM、next-token loss、自回归生成**
3. **9.5 BERT：双向 Encoder、MLM、token classification**
4. **9.3 Seq2Seq：teacher forcing、encoder-decoder**
5. 9.2 RNN/LSTM：保留历史演化与序列建模直觉即可
6. 9.1 NLP 数据流：tokenizer → ids → embedding 必须熟练